In [ ]:
# ==============================================================================
# COMPLETELY INTEGRATED VIIRS FIRMS REFINERY ANALYSIS SCRIPT
# ==============================================================================
# !pip install geopandas shapely pandas matplotlib requests tqdm

import geopandas as gpd
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import requests
import io
from datetime import datetime
from shapely.geometry import Point
from tqdm import tqdm

# --- CONFIGURATION ---
# Ścieżka do Twojego pliku Geopackage z poligonami rafinerii
GPKG_PATH = "path_to_your_refineries.gpkg" 

# Klucz API NASA FIRMS (Darmowy klucz uzyskasz na stronie: https://firms.modaps.eosdis.nasa.gov/api/config/)
# Jeśli nie masz klucza, poniższy domyślny MAPS token z dokumentacji może mieć limity transferu.
FIRMS_API_KEY = "8d5df8354c03b41e8c75df27db32c86e"
# ==============================================================================

def main():
    # -------------------------------------------------------------------------
    # 1. KROK: ŁADOWANIE I PRZYGOTOWANIE GRANIC RAFINERII
    # -------------------------------------------------------------------------
    print("[1/4] Ładowanie granic rafinerii z pliku GPKG...")
    try:
        refineries = gpd.read_file(GPKG_PATH)
    except Exception as e:
        print(f"BŁĄD: Nie można załadować pliku {GPKG_PATH}. Sprawdź ścieżkę. Szczegóły: {e}")
        return

    # Wymagane dopasowanie układu współrzędnych do standardu WGS84 (szeregowe szerokość/długość geograficzna)
    if refineries.crs != "EPSG:4326":
        print(f" Zmiana układu współrzędnych z {refineries.crs} na EPSG:4326...")
        refineries = refineries.to_crs("EPSG:4326")

    # Wyciągamy granice prostokątne obszaru (Bounding Box)
    bbox = refineries.total_bounds
    bbox_str = f"{bbox[0]},{bbox[1]},{bbox[2]},{bbox[3]}"
    print(f" Załadowano {len(refineries)} poligonów. Bounding Box obszaru: {bbox_str}")

    # -------------------------------------------------------------------------
    # 2. KROK: AGREGACJA I POBIERANIE DANYCH HISTORII Z NASA FIRMS API
    # -------------------------------------------------------------------------
    print("\n[2/4] Rozpoczynanie pobierania danych VIIRS (375m) z NASA FIRMS (2012 - Teraz)...")
    
    start_year = 2012  # Rok wejścia operacyjnego satelity Suomi-NPP z sensorem VIIRS
    end_year = datetime.now().year
    all_fires = []

    # Odpytywanie rok po roku chroni przed przepełnieniem bufora i timeoutem API
    for year in tqdm(range(start_year, end_year + 1), desc="Pobieranie rocznych paczek"):
        url = f"https://firms.modaps.eosdis.nasa.gov/api/area/csv/{FIRMS_API_KEY}/VIIRS_SNPP/{bbox_str}/1/{year}"
        
        try:
            response = requests.get(url, timeout=45)
            if response.status_code == 200:
                # Sprawdzenie czy w odpowiedzi są faktyczne nagłówki i dane CSV
                if "latitude" in response.text:
                    df_year = pd.read_csv(io.StringIO(response.text))
                    if not df_year.empty:
                        all_fires.append(df_year)
            elif response.status_code == 429:
                print(f"\n Ostrzeżenie: Przekroczono limit zapytań API (Rate Limit) dla roku {year}.")
            else:
                print(f"\n Ostrzeżenie: API zwróciło kod {response.status_code} dla roku {year}.")
        except Exception as e:
            print(f"\n Błąd połączenia podczas pobierania roku {year}: {e}")
            continue

    if not all_fires:
        print("X Koniec: Nie pobrano żadnych danych punktowych z NASA FIRMS dla tego obszaru.")
        return

    df_raw_fires = pd.concat(all_fires, ignore_index=True)
    print(f" Sukces. Pobrano łącznie {len(df_raw_fires)} ogólnych punktów anomalii termicznych w regionie.")

    # -------------------------------------------------------------------------
    # 3. KROK: FILTROWANIE PRZESTRZENNE (SPATIAL JOIN)
    # -------------------------------------------------------------------------
    print("\n[3/4] Filtrowanie punktów – sprawdzanie, które anomalie leżą wewnątrz rafinerii...")
    
    # Przekształcenie surowych współrzędnych z tabeli na wektorowe punkty geograficzne
    geometry = [Point(xy) for xy in zip(df_raw_fires['longitude'], df_raw_fires['latitude'])]
    fires_gdf = gpd.GeoDataFrame(df_raw_fires, geometry=geometry, crs="EPSG:4326")
    
    # Spatial Join (Przecięcie geometryczne: Punkt WEWNĄTRZ poligonu rafinerii)
    fires_inside = gpd.sjoin(fires_gdf, refineries, predicate='within')
    
    if fires_inside.empty:
        print(" Wynik: Żaden punkt VIIRS nie spadł bezpośrednio wewnątrz granic podanych rafinerii.")
        return
        
    print(f" Sukces! Znaleziono {len(fires_inside)} precyzyjnych trafień wewnątrz granic obiektów.")

    # -------------------------------------------------------------------------
    # 4. KROK: PRZYGOTOWANIE DANYCH CZASOWYCH I WIZUALIZACJA
    # -------------------------------------------------------------------------
    print("\n[4/4] Przygotowywanie osi czasu i generowanie wykresu...")
    
    # Konwersja formatu dat i indeksowanie
    fires_inside['acq_date'] = pd.to_datetime(fires_inside['acq_date'])
    fires_inside = fires_inside.sort_values('acq_date')
    fires_inside.set_index('acq_date', inplace=True)
    
    # Agregacja w interwałach dziennych i miesięcznych
    df_daily = fires_inside.resample('D').size().to_frame(name='fire_count')
    df_monthly = fires_inside.resample('ME').size().to_frame(name='fire_count')

    # Budowanie wykresu
    plt.style.use('seaborn-v0_8-whitegrid' if 'seaborn-v0_8-whitegrid' in plt.style.available else 'default')
    fig, ax = plt.subplots(figsize=(15, 6))
    
    # Rysowanie pionowych słupków (piknięcia pojedynczych dni)
    ax.bar(df_daily.index, df_daily['fire_count'], 
           color='crimson', width=2, alpha=0.7, label='Dzienna liczba anomalii (VIIRS)')
    
    # Rysowanie przerywanej linii trendu (agregacja miesięczna dla wygładzenia)
    ax.plot(df_monthly.index, df_monthly['fire_count'], 
            color='black', linewidth=1.5, linestyle='--', label='Trend miesięczny (Suma)')
    
    # Parametry opisu wykresu
    ax.set_title('Historia wykrytych anomalii termicznych VIIRS na terenie rafinerii (2012 - Teraz)', 
                 fontsize=14, fontweight='bold', pad=15)
    ax.set_xlabel('Rok', fontsize=12)
    ax.set_ylabel('Liczba zarejestrowanych Hotspots (Pikseli)', fontsize=12)
    
    # Dynamiczny zakres osi X
    ax.set_xlim(pd.to_datetime(f"{start_year}-01-01"), pd.to_datetime(datetime.now()))
    
    ax.grid(True, linestyle=':', alpha=0.6)
    ax.legend(loc='upper left', fontsize=11, frameon=True)
    
    plt.tight_layout()
    
    # Wyświetlenie wykresu w notebooku
    plt.show()

if __name__ == "__main__":
    main()